In [1]:
# Installs compatible packages and imports required libraries
!pip -q install scikit-learn==1.5.2 scikeras==0.13.0

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

from scikeras.wrappers import KerasClassifier

np.random.seed(42)
tf.random.set_seed(42)

In [2]:
# Loads and inspects the Pima Indians Diabetes dataset
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

columns = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"
]

data = pd.read_csv(url, names=columns)

print("Dataset shape:", data.shape)
display(data.head())

Dataset shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
# Replaces invalid zero values and fills missing values with medians
zero_columns = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

data[zero_columns] = data[zero_columns].replace(0, np.nan)

for column in zero_columns:
    data[column] = data[column].fillna(data[column].median())

print("Missing values after preprocessing:")
print(data.isnull().sum())

Missing values after preprocessing:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [4]:
# Splits and scales the diabetes data for model training
X = data.drop("Outcome", axis=1)
y = data["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (614, 8)
Testing data: (154, 8)


In [5]:
# Creates the configurable Keras deep learning model
def create_model(
    neurons=16,
    activation="relu",
    dropout_rate=0.0,
    kernel_initializer="glorot_uniform",
    learning_rate=0.001
):
    model = Sequential([
        Dense(
            neurons,
            activation=activation,
            kernel_initializer=kernel_initializer,
            input_shape=(X_train.shape[1],)
        ),
        Dropout(dropout_rate),
        Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [6]:
# Wraps the Keras model for Scikit-learn grid search
keras_model = KerasClassifier(
    model=create_model,
    neurons=16,
    activation="relu",
    dropout_rate=0.0,
    kernel_initializer="glorot_uniform",
    learning_rate=0.001,
    batch_size=16,
    epochs=20,
    verbose=0
)

print("Keras model wrapper created successfully.")

Keras model wrapper created successfully.


In [7]:
# Displays the installed Scikit-learn, SciKeras, and TensorFlow versions
import sklearn
import scikeras
import tensorflow as tf

print("Scikit-learn:", sklearn.__version__)
print("SciKeras:", scikeras.__version__)
print("TensorFlow:", tf.__version__)

Scikit-learn: 1.5.2
SciKeras: 0.13.0
TensorFlow: 2.20.0


In [8]:
# Tunes batch size and training epochs using GridSearchCV
param_grid = {
    "batch_size": [16, 32],
    "epochs": [10, 20]
}

grid_batch_epochs = GridSearchCV(
    keras_model,
    param_grid=param_grid,
    cv=2,
    scoring="accuracy",
    n_jobs=1
)

grid_batch_epochs.fit(X_train, y_train)

print("Best parameters:", grid_batch_epochs.best_params_)
print("Best CV accuracy:", grid_batch_epochs.best_score_)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr

Best parameters: {'batch_size': 16, 'epochs': 20}
Best CV accuracy: 0.757328990228013


In [9]:
# Tunes the learning rate using GridSearchCV
keras_model.set_params(
    batch_size=16,
    epochs=20
)

grid_lr = GridSearchCV(
    keras_model,
    param_grid={"learning_rate": [0.001, 0.01]},
    cv=2,
    scoring="accuracy",
    n_jobs=1
)

grid_lr.fit(X_train, y_train)

print("Best learning rate:", grid_lr.best_params_["learning_rate"])
print("Best CV accuracy:", grid_lr.best_score_)


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr

Best learning rate: 0.01
Best CV accuracy: 0.7638436482084691


In [10]:
# Tunes the weight initialization using GridSearchCV
keras_model.set_params(
    batch_size=16,
    epochs=20,
    learning_rate=0.01
)

grid_init = GridSearchCV(
    keras_model,
    param_grid={"kernel_initializer": ["glorot_uniform", "he_uniform"]},
    cv=2,
    scoring="accuracy",
    n_jobs=1
)

grid_init.fit(X_train, y_train)

print("Best weight initialization:", grid_init.best_params_["kernel_initializer"])
print("Best CV accuracy:", grid_init.best_score_)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr

Best weight initialization: he_uniform
Best CV accuracy: 0.7866449511400652


In [11]:
# Tunes the activation function using GridSearchCV
keras_model.set_params(
    batch_size=16,
    epochs=20,
    learning_rate=0.01,
    kernel_initializer="he_uniform"
)

grid_activation = GridSearchCV(
    keras_model,
    param_grid={"activation": ["relu", "tanh"]},
    cv=2,
    scoring="accuracy",
    n_jobs=1
)

grid_activation.fit(X_train, y_train)

print("Best activation:", grid_activation.best_params_["activation"])
print("Best CV accuracy:", grid_activation.best_score_)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr

Best activation: relu
Best CV accuracy: 0.7687296416938111


In [12]:
# Tunes dropout regularization using GridSearchCV
keras_model.set_params(
    batch_size=16,
    epochs=20,
    learning_rate=0.01,
    kernel_initializer="he_uniform",
    activation="relu"
)

grid_dropout = GridSearchCV(
    keras_model,
    param_grid={"dropout_rate": [0.0, 0.2]},
    cv=2,
    scoring="accuracy",
    n_jobs=1
)

grid_dropout.fit(X_train, y_train)

print("Best dropout rate:", grid_dropout.best_params_["dropout_rate"])
print("Best CV accuracy:", grid_dropout.best_score_)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr

Best dropout rate: 0.0
Best CV accuracy: 0.7833876221498371


In [13]:
# Tunes the number of hidden neurons using GridSearchCV
keras_model.set_params(
    batch_size=16,
    epochs=20,
    learning_rate=0.01,
    kernel_initializer="he_uniform",
    activation="relu",
    dropout_rate=0.0
)

grid_neurons = GridSearchCV(
    keras_model,
    param_grid={"neurons": [16, 32]},
    cv=2,
    scoring="accuracy",
    n_jobs=1
)

grid_neurons.fit(X_train, y_train)

print("Best number of neurons:", grid_neurons.best_params_["neurons"])
print("Best CV accuracy:", grid_neurons.best_score_)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr

Best number of neurons: 32
Best CV accuracy: 0.7801302931596091


In [14]:
# Evaluates the final tuned deep learning model
best_model = grid_neurons.best_estimator_

y_pred = best_model.predict(X_test)
y_pred = np.asarray(y_pred).astype(int).ravel()

test_accuracy = accuracy_score(y_test, y_pred)

print("Final Test Accuracy:", test_accuracy)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Final Test Accuracy: 0.7467532467532467

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.84      0.81       100
           1       0.66      0.57      0.61        54

    accuracy                           0.75       154
   macro avg       0.72      0.71      0.71       154
weighted avg       0.74      0.75      0.74       154



In [15]:
# Displays the final tuned hyperparameters and test accuracy
print("Final Tuned Parameters")
print("Batch Size:", 16)
print("Epochs:", 20)
print("Learning Rate:", 0.01)
print("Weight Initialization:", "he_uniform")
print("Activation:", "relu")
print("Dropout Rate:", 0.0)
print("Hidden Neurons:", 32)
print("Test Accuracy:", test_accuracy)

Final Tuned Parameters
Batch Size: 16
Epochs: 20
Learning Rate: 0.01
Weight Initialization: he_uniform
Activation: relu
Dropout Rate: 0.0
Hidden Neurons: 32
Test Accuracy: 0.7467532467532467


Conclusion: A deep learning model was developed using Keras and Scikit-learn. Grid Search was used to tune batch size, epochs, learning rate, weight initialization, activation function, dropout, and hidden neurons. The final model achieved a test accuracy of 74.68% on the Pima Indians Diabetes dataset.